# RUNG-26b DESIGN (torch only) — RFdiffusion + ProteinMPNN -> binder sequences

RFdiffusion + ProteinMPNN PROVED working on the T4. This notebook generates binder sequences and saves them
to Drive. **No jax/ColabDesign here** (torch and jax cuDNN collide in one kernel) — scoring is a separate
jax-only notebook (`binder_score_colab`). Loads the already-folded target. Run: Cell1 load -> 2 install (~12m) -> 3 smoke -> 4 batch.

In [ ]:
#@title Cell 1 — clone + GPU + load folded target
import os, json
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!nvidia-smi -L
from google.colab import drive; drive.mount('/content/drive')
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json'))
print('target:', M['target_id'], '| hotspot B'+str(M['hotspot']))
print('[CELL 1] OK')

In [ ]:
#@title Cell 2 — install RFdiffusion + ProteinMPNN (torch 2.4 family; NO jax)  [~12 min]
import os, glob, subprocess
def sh(c): r=subprocess.run(c,shell=True,capture_output=True,text=True); (r.returncode and print('  !',c[:60],'\n',(r.stderr or '')[-400:])); return r.returncode
sh('pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124')
sh('pip install -q gemmi')
if not os.path.exists('/content/RFdiffusion'):
    sh('apt-get -qq install -y aria2 >/dev/null')
    sh('git clone -q https://github.com/sokrypton/RFdiffusion.git /content/RFdiffusion')
    sh('git clone -q https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN')
    sh('pip install -q jedi omegaconf hydra-core icecream pyrsistent pynvml decorator')
    sh('pip install -q git+https://github.com/NVIDIA/dllogger#egg=dllogger')
    sh('pip install -q --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html')
    sh('pip install -q --no-dependencies e3nn==0.5.5 opt_einsum_fx')
    sh('cd /content/RFdiffusion/env/SE3Transformer && pip install -q --no-dependencies .')
    os.makedirs('/content/RFdiffusion/models', exist_ok=True)
    sh('cd /content/RFdiffusion/models && aria2c -q -x16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt')
sh('pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124')
RI=glob.glob('/content/RFdiffusion/scripts/run_inference.py')+glob.glob('/content/RFdiffusion/run_inference.py')
print('run_inference.py:', RI[0] if RI else 'NOT FOUND', '| Complex ckpt:', bool(glob.glob('/content/RFdiffusion/models/Complex_base_ckpt.pt')))
print('[CELL 2] done')

In [ ]:
#@title Cell 3 — SMOKE: generate 1 binder sequence (RFdiffusion -> ProteinMPNN)  [~3-5 min]
import os, glob, subprocess, json, gemmi
from google.colab import drive; drive.mount('/content/drive')
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json'))
WORK=M['work']; MUT_PDB=M['mut_pdb']; HOT=f"B{M['hotspot']}"; GL=M['groove_len']; PL=M['pep_len']
RI=(glob.glob('/content/RFdiffusion/scripts/run_inference.py')+glob.glob('/content/RFdiffusion/run_inference.py'))[0]
CONTIG=f"[A1-{GL}/0 B1-{PL}/0 70-90]"
def binder_chain(pdb):
    st=gemmi.read_structure(pdb)
    for c in st[0]:
        n=len(c)
        if n!=GL and n!=PL and 50<=n<=120: return c.name
    return [c.name for c in st[0]][-1]
def rfdiffuse(prefix,n):
    cmd=(f"cd /content/RFdiffusion && python {RI} inference.input_pdb={MUT_PDB} 'contigmap.contigs={CONTIG}' "
         f"'ppi.hotspot_res=[{HOT}]' inference.output_prefix={prefix} inference.num_designs={n} inference.ckpt_override_path=models/Complex_base_ckpt.pt")
    r=subprocess.run(cmd,shell=True,capture_output=True,text=True); print('  rf:',(r.stderr or '')[-500:]); return r
def mpnn(bb):
    ch=binder_chain(bb); out=f'{WORK}/mpnn'; os.makedirs(out,exist_ok=True)
    subprocess.run(f"cd /content/ProteinMPNN && python protein_mpnn_run.py --pdb_path {bb} --pdb_path_chains {ch} --out_folder {out} --num_seq_per_target 1 --sampling_temp 0.1 --seed 1 --batch_size 1",shell=True,capture_output=True,text=True)
    fa=sorted(glob.glob(f'{out}/seqs/*.fa'))
    if not fa: return None
    s=[l.strip() for l in open(fa[-1]) if l.strip() and not l.startswith('>')][-1]
    return s.split('/')[0] if '/' in s else s
rfdiffuse(f'{WORK}/smoke/d',1); bb=sorted(glob.glob(f'{WORK}/smoke/*.pdb'))
print('backbone:', bb[-1] if bb else 'NONE')
if bb:
    seq=mpnn(bb[-1]); print('binder seq:', seq, '| len', len(seq) if seq else 0)
    print('[CELL 3] SMOKE OK — launch Cell 4' if seq else '[CELL 3] MPNN failed')

In [ ]:
#@title Cell 4 — BATCH: generate N binder sequences (time-boxed, resumable)  [~4 h]
max_hours=4.0 #@param {type:'number'}
import os, glob, time, json
from google.colab import drive; drive.mount('/content/drive')
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json')); WORK=M['work']
DES=f'{WORK}/designs'; os.makedirs(DES,exist_ok=True)
t_end=time.time()+max_hours*3600; i=len(glob.glob(f'{DES}/*/seq.json'))
print(f'[CELL 4] resume from {i}; until', time.strftime('%H:%M',time.localtime(t_end)))
while time.time()<t_end:
    did=f'design_{i:04d}'; dd=f'{DES}/{did}'
    if os.path.exists(f'{dd}/seq.json'): i+=1; continue
    os.makedirs(dd,exist_ok=True)
    try:
        rfdiffuse(f'{dd}/bb',1); bb=sorted(glob.glob(f'{dd}/*.pdb'))
        if not bb: raise RuntimeError('no backbone')
        seq=mpnn(bb[-1])
        if not seq: raise RuntimeError('no seq')
        json.dump({'design_id':did,'target':M['target_id'],'sequence':seq}, open(f'{dd}/seq.json','w'))
        print(f'  {did}: len {len(seq)}', flush=True)
    except Exception as e:
        print(f'  {did} FAILED: {e}', flush=True); json.dump({'design_id':did,'error':str(e)}, open(f'{dd}/seq.json','w'))
    i+=1
print(f'[CELL 4] done — {i} sequences. Now run the SCORING notebook (binder_score_colab).')